# Week 3 — Functions & First Contact with Data: Functions Deep Dive
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Write well-structured functions with parameters, defaults, and return values
- Use docstrings for documentation
- Handle edge cases with try/except
- Chain functions together

## Why functions? A story from the Olist warehouse

In the Olist dataset there are **99,441 orders**. For every single one we might ask the same questions:
*What was the total cost (item price + freight)? Was the review positive or negative? Was the delivery fast or slow?*

Imagine answering those questions by copy-pasting the same calculation 99,441 times. One typo and your analysis is wrong — and you would never find it. A **function** lets you write the logic **once**, give it a name, and reuse it everywhere. To process real data at scale, this is the single most important skill we learn today.

### Quick recap before we start

In Weeks 1 and 2 you worked with variables, lists, dictionaries, loops, and `if`/`else` logic. Today we package all of that into reusable **functions**. No data files are needed for this Wednesday session — every example uses real Olist numbers typed directly into the cell. On **Thursday** we will load the actual CSV. Run the tiny cell below to confirm your Python kernel is alive.

In [ ]:
# Warm-up: confirm the kernel runs and recall an f-string from Week 2
total_orders = 99441
print(f"Olist has {total_orders:,} orders to analyse.")   # expected: Olist has 99,441 orders to analyse.

## 1. Function fundamentals: parameters, docstrings, return values

A function is a **named recipe**. You give it some ingredients (the **parameters**), it follows the steps inside, and it hands back a finished dish (the **return value**). Just like a recipe in a cookbook, you write it down once and then cook from it as many times as you like — without rewriting the instructions each time.

The first line `def name(...)` declares the recipe. The triple-quoted text just under it is the **docstring** — a short note explaining what the recipe makes and what each ingredient is for. The `return` statement is what the function hands back to you. We can also give parameters **default values**, so the caller only has to supply them when they want something other than the default.

In [ ]:
def calculate_total_cost(price, freight):
    """
    Calculate the total order cost.

    Args:
        price: item price in R$
        freight: shipping cost in R$
    Returns:
        float: total cost
    """
    return price + freight

print(calculate_total_cost(58.90, 13.29))    # expected: 72.19
print(calculate_total_cost(239.90, 19.93))   # expected: 259.83
print(calculate_total_cost(199.00, 17.87))   # expected: 216.87

### Default parameters

The `classify_review` function below takes a review **score** (1 to 5) and labels it. Notice `positive_threshold=4` and `negative_threshold=2` — these are **defaults**. Because they have a value built in, you can call `classify_review(5)` without supplying them. But if a different analyst decides a "Positive" review needs a score of 5, they can override it: `classify_review(4, positive_threshold=5)`.

In [ ]:
def classify_review(score, positive_threshold=4, negative_threshold=2):
    """Classify review score as Positive, Neutral, or Negative."""
    if score >= positive_threshold:
        return "Positive"
    elif score <= negative_threshold:
        return "Negative"
    else:
        return "Neutral"

for s in [1, 2, 3, 4, 5]:
    print(f"Score {s} -> {classify_review(s)}")
# expected:
# Score 1 -> Negative
# Score 2 -> Negative
# Score 3 -> Neutral
# Score 4 -> Positive
# Score 5 -> Positive

## 2. Functions that return more than one value

Sometimes one answer is not enough. When we analyse a delivery we want **both** the number of days it took **and** a label like "Fast" or "Slow". In Python a function can return several values at once by separating them with commas — this is really a **tuple**. On the calling side we **unpack** the tuple straight into separate variables: `days, status = delivery_analysis(...)`.

This is like a doctor's report that gives you both a number (your temperature) and a verdict ("you have a fever") in one visit. The `try/except` block protects us: if a date string is malformed, instead of crashing we calmly return `(None, "Unknown")`.

In [ ]:
def delivery_analysis(purchase_date_str, delivery_date_str):
    """
    Calculate delivery time and classify it.
    Returns (days, classification) tuple.
    """
    from datetime import datetime
    fmt = "%Y-%m-%d %H:%M:%S"
    try:
        purchase = datetime.strptime(purchase_date_str, fmt)
        delivery = datetime.strptime(delivery_date_str, fmt)
        days = (delivery - purchase).days
        if days <= 7:
            status = "Fast"
        elif days <= 14:
            status = "Normal"
        else:
            status = "Slow"
        return days, status
    except:
        return None, "Unknown"

days, status = delivery_analysis("2017-10-02 10:56:33", "2017-10-10 21:25:13")
print(f"Days: {days}, Status: {status}")   # expected: Days: 8, Status: Normal

days, status = delivery_analysis("2018-07-24 20:41:37", "2018-08-07 15:27:45")
print(f"Days: {days}, Status: {status}")   # expected: Days: 13, Status: Normal

## 3. Going deeper: chaining functions together

The real power appears when the **output of one function becomes the input of another**. This is called *chaining*, and it lets you build a small analysis pipeline out of simple, reliable pieces. Below, `calculate_total_cost` produces a number, and we feed that number straight into a new `cost_tier` function. Each function stays short and easy to test, but together they answer a richer question: *"how expensive is this order, in plain English?"*

This is exactly how professional data work is built — not one giant block of code, but a chain of small, named steps you can trust.

In [ ]:
def cost_tier(total):
    """Label an order total as Budget, Standard, or Premium."""
    if total < 50:
        return "Budget"
    elif total < 200:
        return "Standard"
    else:
        return "Premium"

# Chain: the result of calculate_total_cost flows into cost_tier
order1 = calculate_total_cost(58.90, 13.29)   # 72.19
print(f"Order total R${order1} -> {cost_tier(order1)}")   # expected: Order total R$72.19 -> Standard

order2 = calculate_total_cost(239.90, 19.93)  # 259.83
print(f"Order total R${order2} -> {cost_tier(order2)}")   # expected: Order total R$259.83 -> Premium

### Applying one function to many records

A function only pays off when you run it across **lots** of data. Below we keep a small list of real Olist orders (each is `[price, freight]`) and loop over it, calling our two functions on every row. This is a preview of Thursday: the same pattern will run across all 99,441 orders straight from the CSV file. Notice how the loop body stays short because the *thinking* lives inside the functions.

In [ ]:
# A few real Olist orders as [price, freight] pairs
sample_orders = [
    [58.90, 13.29],
    [239.90, 19.93],
    [199.00, 17.87],
]

for price, freight in sample_orders:
    total = calculate_total_cost(price, freight)
    print(f"R${total:.2f} -> {cost_tier(total)}")
# expected:
# R$72.19 -> Standard
# R$259.83 -> Premium
# R$216.87 -> Premium

## 4. Common mistakes with functions

Two errors trip up almost every beginner. The first is **forgetting `return`** — without it, your function quietly hands back `None`, and the rest of your code breaks in confusing ways. The second is **calling before defining** — Python reads top to bottom, so a function must be defined *above* the line that uses it. Study the comments below before you write your own functions.

In [ ]:
# == COMMON MISTAKE 1: forgetting return ==
# WRONG -- this function PRINTS but returns None:
# def add_bad(a, b):
#     a + b               # computed, then thrown away!
# result = add_bad(58.90, 13.29)
# print(result)           # None  -- not 72.19

# CORRECT -- you must RETURN the value to use it later:
def add_good(a, b):
    return a + b
result = add_good(58.90, 13.29)
print(result)             # expected: 72.19

# == COMMON MISTAKE 2: calling before defining ==
# WRONG -- this raises NameError because greet is defined LATER:
# print(greet("Olist"))   # NameError: name 'greet' is not defined
# def greet(name):
#     return f"Hello {name}"

# CORRECT -- define first, then call:
def greet(name):
    return f"Hello {name}"
print(greet("Olist"))     # expected: Hello Olist

## 5. Mini-challenge: build `format_currency`

Write a function `format_currency(amount, currency="R$")` that turns a number into a tidy money string with thousands separators.

**Expected output:**
```
R$5,202,955.05
$1,234.56
```

*Hint:* an f-string can format numbers with commas and 2 decimals using `:,.2f` — e.g. `f"{amount:,.2f}"`.

⏱ ~5 minutes

In [ ]:
# Mini-challenge: complete the function below

def format_currency(amount, currency="R$"):
    """Return amount as a formatted currency string."""
    # Your code here -- build and return the formatted string
    pass

# Test cases (do not change):
print(format_currency(5202955.05))        # target: R$5,202,955.05
print(format_currency(1234.56, "$"))      # target: $1,234.56

## Wednesday Group Exercise: Build the Olist Toolkit

Write these 4 functions using what you have learned:

```python
# Function 1: summarise_orders(status_list, count_list)
# Takes two lists, returns a dict: {status: count}
# Test with the 8 Olist statuses and counts

# Function 2: top_n_items(data_dict, n=5)
# Takes a dict {key: numeric_value}, returns list of top n (key, value) tuples
# Test: top_n_items({"SP":41746,"RJ":12852,"MG":11635,"RS":5466,"PR":5045}, 3)
# Expected: [("SP",41746), ("RJ",12852), ("MG",11635)]

# Function 3: format_currency(amount, currency="R$")
# Returns formatted string: "R$1,234.56"
# Test: format_currency(5202955.05) -> "R$5,202,955.05"

# Function 4: safe_divide(numerator, denominator, default=0.0)
# Returns numerator/denominator, returns default if denominator is 0
# Test: safe_divide(96478, 99441) -> 0.9702...
# Test: safe_divide(100, 0) -> 0.0
```

In [ ]:
# Group Exercise -- complete all 4 functions. Input data is pre-defined; do not change it.

# Input data for testing (do not change):
statuses = ["delivered", "shipped", "canceled", "unavailable",
            "invoiced", "processing", "created", "approved"]
counts   = [96478, 1107, 625, 609, 314, 301, 5, 2]
states   = {"SP": 41746, "RJ": 12852, "MG": 11635, "RS": 5466, "PR": 5045}


# Function 1: summarise_orders(status_list, count_list) -> dict
def summarise_orders(status_list, count_list):
    # Your code here
    pass


# Function 2: top_n_items(data_dict, n=5) -> list of (key, value) tuples
def top_n_items(data_dict, n=5):
    # Your code here
    pass


# Function 3: format_currency(amount, currency="R$") -> str
def format_currency(amount, currency="R$"):
    # Your code here
    pass


# Function 4: safe_divide(numerator, denominator, default=0.0) -> float
def safe_divide(numerator, denominator, default=0.0):
    # Your code here
    pass


# --- Tests (run after completing the functions) ---
# print(summarise_orders(statuses, counts))
# print(top_n_items(states, 3))           # [("SP",41746), ("RJ",12852), ("MG",11635)]
# print(format_currency(5202955.05))      # R$5,202,955.05
# print(safe_divide(96478, 99441))        # 0.9702...
# print(safe_divide(100, 0))              # 0.0

## Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Define a function | `def name(params):` | `def calculate_total_cost(price, freight):` |
| Return a value | `return value` | `return price + freight` |
| Docstring | `"""..."""` under `def` | documents what the function does |
| Default parameter | `param=default` | `classify_review(score, positive_threshold=4)` |
| Multiple return values | `return a, b` | `days, status = delivery_analysis(...)` |
| Handle errors | `try: ... except: ...` | returns `(None, "Unknown")` on bad input |
| Chain functions | `f(g(x))` | `cost_tier(calculate_total_cost(p, f))` |

---
**Coming up Thursday**: *Reading Real Data with Python* — we open the actual `olist_orders_dataset.csv` (all 99,441 rows) with the `csv` module and apply the functions you built today to real data.